# 02. XYZ-анализ стабильности спроса

## Цель и контекст
Цель данного этапа — классифицировать товары по стабильности спроса на основе помесячных продаж. Результаты XYZ-анализа будут использованы для определения стратегии прогнозирования и управления запасами.

## Загрузка и подготовка данных

In [43]:
import pandas as pd
from pathlib import Path

data_path = Path.cwd().parent.parent / "data" / "clean"
sales_data_path = data_path / "sales_data.csv"

sales_data = pd.read_csv(sales_data_path)

sales_data.head()

,product,sku,qty,unit,month
0,КОНФ ВЕС Столичные,КО01828,96.613,кг,2025-08
1,КОНФ ВЕС Сибирский Сувенир,НС07823,31.901,кг,2025-08
2,КОНФ ВЕС Тамбовский волк Люкс,ТК07914,26.931,кг,2025-08
3,КОНФ ВЕС БУТЫЛОЧКИ С КОНЬЯКОМ,ЯП24671,22.990,кг,2025-08
4,КОНФ ВЕС Столичные любимые,ВО13951,39.748,кг,2025-08


## Расчёт коэффициента вариации (CV)

CV = относительный шум спроса

Если сказать по-человечески:

«На сколько процентов в среднем спрос отклоняется от своего нормального уровня»

Это не разброс сам по себе, а разброс относительно масштаба товара.

In [44]:
cv_data = sales_data.groupby("sku")["qty"].agg(["mean", "std", "count"])
# Сохраняем статистику и сразу формируем таблицу с count
cv_result = cv_data.copy()
cv_result["cv"] = cv_data["std"] / cv_data["mean"]
cv_result = cv_result.reset_index()
cv_result.tail()

,sku,mean,std,count,cv
720,рф24470,2.000000,NaN,1,NaN
721,слв 1035,12.666667,6.164414,9,0.486664
722,слв6002,3.666667,2.179449,9,0.594395
723,слв6004,3.000000,1.511858,8,0.503953
724,слв6006,3.333333,1.581139,9,0.474342


In [45]:
cv_data["std"].head()

sku
0104          NaN
15033    2.869379
38010    3.003029
38015    4.272534
38030    1.763834
Name: std, dtype: float64

Общая статистика по таблице CV

In [46]:
cv_result["cv"].describe()

count    674.000000
mean       0.513150
std        0.211301
min        0.000000
25%        0.382700
50%        0.487900
75%        0.637725
max        1.321358
Name: cv, dtype: float64

Строки где CV = 0

In [47]:
cv_result["cv"].isna().sum()

np.int64(51)

Товари с самым маленьким коэфом

In [48]:
cv_result.sort_values("cv").head(10)

,sku,mean,std,count,cv
96,ББ25615,6.000000,0.000000,2,0.000000
415,РФ14503,2.000000,0.000000,2,0.000000
465,РФ18954,8.000000,0.000000,2,0.000000
614,СР13939,7.000000,0.000000,2,0.000000
520,РФ25091,20.000000,0.000000,2,0.000000
519,РФ25088,9.833333,0.408248,6,0.041517
412,РФ13537,12.666667,0.577350,3,0.045580
414,РФ13638,10.500000,0.707107,2,0.067344
124,ЙО19104,20.000000,1.414214,2,0.070711
576,СМ06262,10.000000,0.707107,5,0.070711


Товары с самым большим коэфом

In [49]:
cv_result.sort_values("cv", ascending=False).head(10)

,sku,mean,std,count,cv
288,КО25456,32.333333,42.723920,3,1.321358
279,КО24995,28.800000,37.211557,5,1.292068
296,КО25985,2.011333,2.357026,3,1.171872
643,ЮК03584,23.000000,26.851443,3,1.167454
314,НС02133,4.943500,5.613721,2,1.135576
72,ББ23412,26.818182,30.432937,11,1.134787
94,ББ25578,24.000000,26.457513,3,1.102396
129,ЙО22643,9.000000,9.899495,2,1.099944
558,РФ25387,7.750000,8.381527,4,1.081487
317,НС09082,9.000000,9.695360,6,1.077262


Проверка распределения count

In [50]:
# Теперь столбец 'count' уже присутствует в cv_result
cv_result["count"].describe()

count    725.000000
mean       7.451034
std        3.386990
min        1.000000
25%        5.000000
50%        9.000000
75%       11.000000
max       11.000000
Name: count, dtype: float64

Создание новых переменных для рпазделения на XYZ_stat и на XYZ_operational

In [51]:
cv_operational = cv_result
cv_stat = cv_result

## Классификация товаров по XYZ (stat)
В этом блоке мы классифицируем товары по XYZ с точки зрения идеальной статистики. Делаем это для прогнозирования рисков

In [52]:
def make_classifier(x_thresh: float, y_thresh: float, min_months: int = 3):
    """Фабрика функций-классификаторов XYZ.

    Возвращает функцию, которую можно применять к строке DataFrame (row),
    она использует пороги x_thresh и y_thresh и минимальное количество месяцев.
    """
    def classify(row):
        cv = row["cv"]
        count = row["count"]

        # Проверяем количество месяцев
        if count < min_months:
            return "Unknown"

        # Проверяем CV
        if pd.isna(cv):
            return "Unknown"
        elif cv <= x_thresh:
            return "X - Стабильный"
        elif cv <= y_thresh:
            return "Y - Средний"
        else:
            return "Z - Нестабильный"

    return classify

# Классификатор для "stat" (более строгие пороги)
stat_clf = make_classifier(x_thresh=0.1, y_thresh=0.25, min_months=3)
cv_stat["xyz_stat"] = cv_stat.apply(stat_clf, axis=1)
cv_stat["xyz_stat"].value_counts()

xyz_stat
Z - Нестабильный    582
Unknown              94
Y - Средний          45
X - Стабильный        4
Name: count, dtype: int64

## Классификация товаров по XYZ (operational)
В этом блоке мы так же классифицируем товары по XYZ, но повышаем коэфициенты классификации для того, чтобы мы могли прогнозировать спрос на большее количество товаров (~47%), что лучше для бизнеса, но более опасно, так как у нас неготорые товары из Z преходят в Y и из Y в X.

Для дальнейшего прогнозирования будем пользоваться этими вычислениями, блок stat будем использовать для анализа рисков.

In [53]:
# Классификатор для "operational" (более мягкие пороги)
operational_clf = make_classifier(x_thresh=0.25, y_thresh=0.5, min_months=3)
cv_operational["xyz_operational"] = cv_operational.apply(operational_clf, axis=1)
cv_operational["xyz_operational"].value_counts()

xyz_operational
Z - Нестабильный    301
Y - Средний         281
Unknown              94
X - Стабильный       49
Name: count, dtype: int64

## Анализ распределения товаров

## Выводы и решения

In [55]:
# Подготовка данных для сохранения (stat и operational)
xyz_stat = cv_stat[["sku", "count", "cv", "xyz_stat"]].copy()
xyz_stat = xyz_stat.rename(columns={
    "sku": "Артикул",
    "count": "Количество месяцев",
    "cv": "CV коэффициент",
    "xyz_stat": "Класс товара"
})

xyz_operational = cv_operational[["sku", "count", "cv", "xyz_operational"]].copy()
xyz_operational = xyz_operational.rename(columns={
    "sku": "Артикул",
    "count": "Количество месяцев",
    "cv": "CV коэффициент",
    "xyz_operational": "Класс товара"
})

# Сохраняем в CSV
output_path = Path.cwd().parent.parent / "data" / "clean"
output_path.mkdir(parents=True, exist_ok=True)

output_file_stat = output_path / "xyz_stat_result.csv"
xyz_stat.to_csv(output_file_stat, index=False, encoding="utf-8")
print(f"✅ Результаты XYZ (stat) сохранены в: {output_file_stat}")
print(xyz_stat.head(5))

print("\n---\n")

output_file_oper = output_path / "xyz_operational_result.csv"
xyz_operational.to_csv(output_file_oper, index=False, encoding="utf-8")
print(f"✅ Результаты XYZ (operational) сохранены в: {output_file_oper}")
print(xyz_operational.head(5))


✅ Результаты XYZ (stat) сохранены в: /Users/aleksejsuharev/Desktop/Аленка/candy_forecast/data/clean/xyz_stat_result.csv
  Артикул  Количество месяцев  CV коэффициент      Класс товара
0    0104                   1             NaN           Unknown
1   15033                  10        0.372647  Z - Нестабильный
2   38010                  11        0.635256  Z - Нестабильный
3   38015                  11        0.367171  Z - Нестабильный
4   38030                  10        0.293972  Z - Нестабильный

---

✅ Результаты XYZ (operational) сохранены в: /Users/aleksejsuharev/Desktop/Аленка/candy_forecast/data/clean/xyz_operational_result.csv
  Артикул  Количество месяцев  CV коэффициент      Класс товара
0    0104                   1             NaN           Unknown
1   15033                  10        0.372647       Y - Средний
2   38010                  11        0.635256  Z - Нестабильный
3   38015                  11        0.367171       Y - Средний
4   38030                  10       